# Language Modeling Objectives: cross-entropy by hand, then watch a tiny LM learn

You have a trillion words of raw text and not one label. Yet GPT, BERT, and T5 all learn language from it. The trick is the **language modeling objective**: hide part of the text, predict it, and score how surprised the model is by the truth. The text *is* the label — that's self-supervision, and it's where ~all of an LLM's capability comes from.

This notebook makes the **causal** (next-token) objective concrete, in small pieces:

1. compute next-token **cross-entropy** and **perplexity by hand** on a toy vocabulary, so the arithmetic is checkable;
2. build and print the **causal mask** — the rule that position $t$ can't peek at the future;
3. train a tiny causal LM a few steps and **watch loss and perplexity drop** toward their floor.

**By the end** you'll have computed the loss the way the math says, seen the mask that makes prediction honest, and watched maximum likelihood drive perplexity toward 1 — all from scratch, on CPU, in seconds.

## The objective: predict the missing token

The probability of a sentence factorizes **exactly** by the chain rule:

$$p(x_1,\dots,x_n) = \prod_{t=1}^{n} p(x_t \mid x_{<t})$$

The model's only job is to learn each conditional $p(x_t \mid x_{<t})$ — *given everything so far, what comes next?* We score it with **cross-entropy** (the negative log-probability of the true token), average over positions, and read it as **perplexity** $= e^{\text{loss}}$: the effective number of equally-likely tokens the model feels it's choosing among.

Everything below is that one idea, made runnable.

> **Step 1 — set up.** Pick the device at runtime (CUDA → MPS → CPU so it runs anywhere), seed for reproducibility, and define a 5-token toy vocabulary. Everything in the by-hand and mask demos is small enough to read by eye; only the tiny training model uses real tensors.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("compute device available:", device, "— the by-hand math is exact on any device; the tiny training trace below runs on CPU for a reproducible curve")
print("torch:", torch.__version__)

torch.manual_seed(0)
VOCAB = ["the", "cat", "sat", "on", "mat"]   # a 5-token toy vocabulary
V = len(VOCAB)
print("vocab size V =", V)

compute device available: mps — the by-hand math is exact on any device; the tiny training trace below runs on CPU for a reproducible curve
torch: 2.12.0
vocab size V = 5


## Cross-entropy and perplexity, by hand

Before any model, settle the loss arithmetic on a single prediction so you can check it yourself.

> **Step 2 — one prediction, scored by hand.** Suppose the model predicted the distribution below over the next token, and the truth is `"sat"` (id 2). Cross-entropy is just $-\log p(\text{true})$, and perplexity is $e^{\text{loss}}$. Because it gave the true token exactly 0.50, the loss should come out to $-\log 0.5 = 0.6931$ nats and perplexity to exactly **2.0** — "as uncertain as a fair coin." We also show the cost of a *confident-but-wrong* guess: probability 0.05 on the truth costs far more.

In [2]:
probs = torch.tensor([0.10, 0.20, 0.50, 0.15, 0.05])   # model's predicted next-token distribution (sums to 1.0)
true_id = VOCAB.index("sat")                            # the token that actually came next

loss_by_hand = -math.log(probs[true_id].item())        # cross-entropy = -log p(true token)
ppl_by_hand = math.exp(loss_by_hand)                    # perplexity = exp(loss)
print(f"p(true='sat') = {probs[true_id]:.2f}")
print(f"cross-entropy = -log({probs[true_id]:.2f}) = {loss_by_hand:.4f} nats")
print(f"perplexity    = exp({loss_by_hand:.4f})    = {ppl_by_hand:.4f}")
print(f"if it had said p=0.05 instead: loss = {-math.log(0.05):.4f} (much worse)")

p(true='sat') = 0.50
cross-entropy = -log(0.50) = 0.6931 nats
perplexity    = exp(0.6931)    = 2.0000
if it had said p=0.05 instead: loss = 2.9957 (much worse)


> **Step 3 — the same loss via `F.cross_entropy`.** PyTorch's `cross_entropy` takes **logits** (not probabilities) and the **true token id** (not a one-hot), applies softmax internally, and returns $-\log p(\text{true})$. We feed it logits whose softmax is the distribution above and confirm we get the same **0.6931** — proof that the library and the by-hand formula are the same thing. (Real models always pass logits, never probabilities, for numerical stability.)

In [3]:
logits = torch.log(probs).unsqueeze(0)      # logits whose softmax == probs; shape [1, V]
label = torch.tensor([true_id])             # the true token id, shape [1]
print("logits shape:", tuple(logits.shape), " label shape:", tuple(label.shape))
loss_torch = F.cross_entropy(logits, label)
print(f"F.cross_entropy loss = {loss_torch.item():.4f}  (matches the by-hand {loss_by_hand:.4f})")

logits shape: (1, 5)  label shape: (1,)
F.cross_entropy loss = 0.6931  (matches the by-hand 0.6931)


## The causal mask: no peeking at the future

For next-token prediction to be honest, position $t$ must not see tokens $t+1, t+2, \dots$ — otherwise predicting the next token is trivial (it's right there). The **causal mask** enforces exactly that.

> **Step 4 — build and read the mask.** A lower-triangular matrix of 1s: row $t$ (a query position) has 1s in columns $0..t$ and 0s above the diagonal. Read it row by row — position 0 sees only itself; position 3 sees all four. The zeros above the diagonal are the future, blocked. In a real transformer these 0s become $-\infty$ added to the attention scores *before* softmax, so future positions get zero attention weight.

In [4]:
T = 4
causal = torch.tril(torch.ones(T, T))       # lower-triangular: 1 = may attend, 0 = blocked
print("causal mask shape:", tuple(causal.shape))
print("causal mask (1=can attend, 0=blocked):")
print(causal.int().numpy())
print("\nposition 0 attends to", int(causal[0].sum()), "token(s);",
      "position 3 attends to", int(causal[3].sum()), "tokens")

causal mask shape: (4, 4)
causal mask (1=can attend, 0=blocked):
[[1 0 0 0]
 [1 1 0 0]
 [1 1 1 0]
 [1 1 1 1]]

position 0 attends to 1 token(s); position 3 attends to 4 tokens


## Watch a tiny causal LM learn

Now put the objective to work: train a tiny model on one repeated sentence and watch loss and perplexity fall as it learns the next-token pattern.

> **Step 5 — a tiny causal LM.** The smallest model that has next-token logits: embed each token, mix with one nonlinear layer, then project to one logit per vocabulary token. It has no real attention — we only need it to show that the *objective* drives loss down. The forward pass returns logits shaped `[batch, seq, vocab]`; keep that shape in mind, it's what the loss consumes.

In [5]:
class TinyLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, 16)       # token id -> 16-dim vector
        self.ff = nn.Linear(16, 16)          # one mixing layer
        self.head = nn.Linear(16, V)         # project to one logit per vocab token
    def forward(self, x):
        h = torch.tanh(self.ff(self.emb(x)))
        return self.head(h)                  # logits: [batch, seq, vocab]

sentence = torch.tensor([[0, 1, 2, 3, 4]])   # 'the cat sat on mat' as ids, shape [1, 5]
model = TinyLM()
print("input ids shape:", tuple(sentence.shape))
print("logits shape:   ", tuple(model(sentence).shape), "  -> [batch, seq, vocab]")

input ids shape: (1, 5)
logits shape:    (1, 5, 5)   -> [batch, seq, vocab]


> **Step 6 — the SHIFT, the easiest LM bug to get wrong.** Position $t$ predicts token $t+1$, so the logits and labels must be **shifted by one**: `logits[:, :-1]` (drop the last position, which has no next token) lines up with `labels[:, 1:]` (drop the first, which is never a target). With 5 tokens that's **4 predictions**. Print the shapes and confirm they match before trusting any loss number.

In [6]:
logits = model(sentence)                                # [1, 5, V]
shifted_logits = logits[:, :-1].reshape(-1, V)          # [4, V] — predictions for positions 0..3
shifted_labels = sentence[:, 1:].reshape(-1)            # [4]    — true tokens at positions 1..4
print("shifted_logits:", tuple(shifted_logits.shape), " shifted_labels:", tuple(shifted_labels.shape))
print("predicting tokens", shifted_labels.tolist(), "= ", [VOCAB[i] for i in shifted_labels.tolist()])
loss0 = F.cross_entropy(shifted_logits, shifted_labels)
print(f"initial loss = {loss0.item():.4f}  ->  perplexity = {math.exp(loss0.item()):.4f}")

shifted_logits: (4, 5)  shifted_labels: (4,)
predicting tokens [1, 2, 3, 4] =  ['cat', 'sat', 'on', 'mat']
initial loss = 1.7148  ->  perplexity = 5.5554


> **Step 7 — train, and watch perplexity collapse toward 1.** Run Adam for a couple hundred steps on this one sentence and print loss and perplexity every 40 steps. Because there's only one sentence, a good model can *memorize* it — so perplexity should fall from "uncertain among several tokens" toward **~1.0** ("certain of the right next token"). That downward curve *is* maximum likelihood working: every step raises $p(\text{true})$, which lowers $-\log p$.

In [7]:
torch.manual_seed(0)
model = TinyLM()                                         # fresh model for a clean curve
opt = torch.optim.Adam(model.parameters(), lr=0.05)

print(f"{'step':>4} | {'loss':>7} | {'perplexity':>10}")
print("-" * 28)
for step in range(0, 201, 40):
    for _ in range(40 if step else 1):                  # train in chunks of 40 steps
        logits = model(sentence)                        # [1, 5, V]
        # SHIFT: position t predicts token t+1
        loss = F.cross_entropy(logits[:, :-1].reshape(-1, V), sentence[:, 1:].reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"{step:>4} | {loss.item():>7.4f} | {math.exp(loss.item()):>10.4f}")

step |    loss | perplexity
----------------------------
   0 |  1.7148 |     5.5554
  40 |  0.0002 |     1.0002
  80 |  0.0001 |     1.0001
 120 |  0.0001 |     1.0001
 160 |  0.0001 |     1.0001
 200 |  0.0001 |     1.0001


## Try it yourself

Before you run it, **predict**: we trained on a *single* repeated sentence, so the model drove perplexity to ~1.0 by **memorizing** it. If you instead train on **two different** sentences that share a prefix but diverge — `the cat sat on mat` *and* `the cat sat on rug` — does the final perplexity settle **at 1.0**, **above 1.0**, or **below 1.0**?

Then run the cell and check.

*Hint:* after `the cat sat on`, the true next token is genuinely ambiguous — sometimes `mat`, sometimes `rug` — so even a perfect model can't put probability 1 on either. The best it can do is ~0.5 each, which floors perplexity **above** 1.0. That floor is the **irreducible entropy** of the data — the real reason no LM reaches PPL 1 on real text.

In [8]:
# 'rug' needs a 6th vocab id; extend the vocab to 6 for this experiment only.
V2 = 6                                                   # the, cat, sat, on, mat, rug
batch = torch.tensor([[0, 1, 2, 3, 4],                   # the cat sat on mat
                      [0, 1, 2, 3, 5]])                  # the cat sat on rug

class TinyLM2(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V2, 16); self.ff = nn.Linear(16, 16); self.head = nn.Linear(16, V2)
    def forward(self, x):
        return self.head(torch.tanh(self.ff(self.emb(x))))

torch.manual_seed(0)
m2 = TinyLM2(); opt2 = torch.optim.Adam(m2.parameters(), lr=0.05)
for _ in range(400):
    lg = m2(batch)
    loss = F.cross_entropy(lg[:, :-1].reshape(-1, V2), batch[:, 1:].reshape(-1))
    opt2.zero_grad(); loss.backward(); opt2.step()
print(f"two divergent sentences -> final loss {loss.item():.4f}, perplexity {math.exp(loss.item()):.4f}")
print("floored ABOVE 1.0: the last token is genuinely ambiguous, so it can't be predicted with certainty.")

two divergent sentences -> final loss 0.1733, perplexity 1.1892
floored ABOVE 1.0: the last token is genuinely ambiguous, so it can't be predicted with certainty.


## What we saw

- **Cross-entropy is just $-\log p(\text{true})$**, and `F.cross_entropy` (on logits + token ids) computes exactly that — we matched both to **0.6931** for a 0.50-probability prediction, whose perplexity is **2.0** (a fair coin).
- **The causal mask is lower-triangular** — position $t$ attends only to $0..t$, never the future. That's what makes next-token prediction honest.
- **The shift** (`logits[:, :-1]` vs `labels[:, 1:]`) lines up each position with the token it predicts — the single easiest LM-loss bug to get wrong.
- **Training drove perplexity toward 1.0** on one memorized sentence — maximum likelihood at work — but **floored above 1.0** the moment the data had genuine ambiguity, which is the irreducible entropy real text always has — here ~1.19, a touch above the theoretical ~1.09 because 400 steps don't fully zero even the deterministic positions; train longer and it settles toward ~1.09, but never to 1.0.

**What we skipped** (and where to read it): the **masked** objective behind BERT (predict random blanks from both sides), the **span-corruption** objective behind T5, and *why* causal LM became the foundation of generative AI. All of that is in the [Language Modeling Objectives concept page](../01-Language-Modeling-Objectives.md), with its [curated references](../01-Language-Modeling-Objectives.references.md).